In [4]:
import pandas as pd
import numpy as np

# Establecer una semilla para reproducibilidad
np.random.seed(42)

# Número de transacciones
n_samples = 5000

# Generar datos aleatorios
monto_pesos = np.random.uniform(100, 100000, n_samples).round(2)
hora_dia = np.random.randint(0, 24, n_samples)
distancia_residencia = np.random.uniform(0.1, 500, n_samples).round(2)
ingreso_mensual = np.random.uniform(1000, 500000, n_samples).round(2)

# Generar la etiqueta de fraude (simulando cierto patrón para que el GP tenga algo que aprender)
# Por ejemplo, mayor monto, hora de madrugada, o gran distancia podría ser fraude
fraude = np.zeros(n_samples, dtype=int)

# Simular algunos patrones de fraude:
# Fraude 1: montos muy altos y horas de madrugada
fraude[(monto_pesos > 80000) & (hora_dia < 6)] = 1
# Fraude 2: ingreso mensual bajo y monto alto con distancia alta
fraude[(ingreso_mensual < 10000) & (monto_pesos > 50000) & (distancia_residencia > 200)] = 1
# Fraude 3: Una pequeña proporción de fraude aleatorio para añadir ruido
fraude[np.random.rand(n_samples) < 0.02] = 1

df = pd.DataFrame({
    'monto_pesos': monto_pesos,
    'hora_dia': hora_dia,
    'distancia_residencia': distancia_residencia,
    'ingreso_mensual': ingreso_mensual,
    'fraude': fraude
})

print(f"Total de transacciones: {len(df)}")
print(f"Transacciones con fraude: {df['fraude'].sum()}")
print(f"Proporción de fraude: {df['fraude'].mean():.2%}")

display(df.head())


Total de transacciones: 5000
Transacciones con fraude: 364
Proporción de fraude: 7.28%


,monto_pesos,hora_dia,distancia_residencia,ingreso_mensual,fraude
0,37516.56,18,436.01,99651.25,0
1,95076.36,7,49.12,182667.37,0
2,73226.19,6,218.21,333817.80,0
3,59905.98,15,481.16,224083.72,0
4,15686.26,18,417.09,467095.85,0


In [5]:
# Instalar gplearn si no está instalado
!pip install gplearn

In [6]:
from sklearn.model_selection import train_test_split
from gplearn.genetic import SymbolicClassifier
from sklearn.metrics import accuracy_score, classification_report

# Preparar los datos
X = df.drop('fraude', axis=1)
y = df['fraude']

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Inicializar y entrenar el clasificador simbólico
# Se pueden ajustar muchos parámetros aquí (generaciones, población, funciones, etc.)
# Para este ejemplo, usaremos una configuración básica.

gp_classifier = SymbolicClassifier(population_size=5000, # Tamaño de la población de programas
                                   generations=100,       # Número de generaciones para evolucionar
                                   tournament_size=20,   # Número de programas para seleccionar en cada torneo
                                   stopping_criteria=0.99, # Detener si la precisión de entrenamiento alcanza esto
                                   p_crossover=0.7,      # Probabilidad de cruce
                                   p_subtree_mutation=0.1, # Probabilidad de mutación de subárbol
                                   p_hoist_mutation=0.05, # Probabilidad de mutación de elevación
                                   p_point_mutation=0.1,  # Probabilidad de mutación puntual
                                   max_samples=0.9,      # Proporción de muestras para entrenar cada generación
                                   verbose=1,            # Mostrar el progreso durante el entrenamiento
                                   random_state=42,      # Semilla para reproducibilidad
                                   n_jobs=-1             # Usar todos los núcleos disponibles
                                  )

gp_classifier.fit(X_train, y_train)

print("\n--- Modelo de Programación Genética Entrenado ---")
print(f"Mejor programa encontrado:\n{gp_classifier._program}")

# Evaluar el modelo en el conjunto de prueba
y_pred = gp_classifier.predict(X_test)

print("\n--- Evaluación del Modelo ---")
print(f"Precisión en el conjunto de prueba: {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

# Puedes ver la estructura del árbol del mejor programa
# print(gp_classifier._program.export_graphviz())


    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    30.42          15.1574        5          0.25808         0.256064     24.08m

--- Modelo de Programación Genética Entrenado ---
Mejor programa encontrado:
mul(div(X1, 0.399), -0.119)

--- Evaluación del Modelo ---
Precisión en el conjunto de prueba: 0.9273

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1391
           1       0.00      0.00      0.00       109

    accuracy                           0.93      1500
   macro avg       0.46      0.50      0.48      1500
weighted avg       0.86      0.93      0.89      1500



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [7]:
# Instalar imblearn si no está instalado
!pip install imbalanced-learn

In [8]:
from imblearn.over_sampling import RandomOverSampler

print(f"Forma original de X_train: {X_train.shape}")
print(f"Distribución original de y_train:\n{y_train.value_counts()}")

# Inicializar RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Aplicar sobremuestreo a los datos de entrenamiento
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

print(f"\nForma de X_train después del sobremuestreo: {X_train_resampled.shape}")
print(f"Distribución de y_train después del sobremuestreo:\n{y_train_resampled.value_counts()}")

Forma original de X_train: (3500, 4)
Distribución original de y_train:
fraude
0    3245
1     255
Name: count, dtype: int64

Forma de X_train después del sobremuestreo: (6490, 4)
Distribución de y_train después del sobremuestreo:
fraude
0    3245
1    3245
Name: count, dtype: int64


In [9]:
print("Reentrenando el modelo de Programación Genética con datos balanceados...")

# Reinicializar y entrenar el clasificador simbólico con los datos rebalanceados
# Se mantienen los mismos parámetros que antes
gp_classifier_resampled = SymbolicClassifier(population_size=5000,
                                           generations=100,
                                           tournament_size=20,
                                           stopping_criteria=0.99,
                                           p_crossover=0.7,
                                           p_subtree_mutation=0.1,
                                           p_hoist_mutation=0.05,
                                           p_point_mutation=0.1,
                                           max_samples=0.9,
                                           verbose=1,
                                           random_state=42,
                                           n_jobs=-1)

gp_classifier_resampled.fit(X_train_resampled, y_train_resampled)

print("\n--- Modelo de Programación Genética Reentrenado con Datos Balanceados ---")
print(f"Mejor programa encontrado:\n{gp_classifier_resampled._program}")

# Evaluar el modelo en el conjunto de prueba (no balanceado) original
y_pred_resampled = gp_classifier_resampled.predict(X_test)

print("\n--- Evaluación del Modelo Reentrenado ---")
print(f"Precisión en el conjunto de prueba (original): {accuracy_score(y_test, y_pred_resampled):.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_resampled))

Reentrenando el modelo de Programación Genética con datos balanceados...
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    30.42          12.8863       15         0.638708         0.645617     22.95m

--- Modelo de Programación Genética Reentrenado con Datos Balanceados ---
Mejor programa encontrado:
div(sub(add(X0, X1), mul(X1, 0.158)), mul(add(X3, X2), mul(0.898, X1)))

--- Evaluación del Modelo Reentrenado ---
Precisión en el conjunto de prueba (original): 0.0727

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1391
           1       0.07      1.00      0.14       109

    accuracy                           0.07      1500
   macro avg       0.04      0.50      0.07      1500
weighted avg       0.01    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [10]:
from sklearn.metrics import accuracy_score, classification_report

# Obtener las probabilidades de predicción para la clase positiva (fraude)
y_proba = gp_classifier_resampled.predict_proba(X_test)[:, 1]

# Probar con un umbral más alto, por ejemplo, 0.7
custom_threshold = 0.7
y_pred_thresholded = (y_proba >= custom_threshold).astype(int)

print(f"\n--- Evaluación del Modelo Reentrenado con Umbral de {custom_threshold} ---")
print(f"Precisión en el conjunto de prueba (original) con umbral {custom_threshold}: {accuracy_score(y_test, y_pred_thresholded):.4f}")
print("\nReporte de Clasificación con umbral ajustado:")
print(classification_report(y_test, y_pred_thresholded))


--- Evaluación del Modelo Reentrenado con Umbral de 0.7 ---
Precisión en el conjunto de prueba (original) con umbral 0.7: 0.8920

Reporte de Clasificación con umbral ajustado:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1391
           1       0.20      0.17      0.18       109

    accuracy                           0.89      1500
   macro avg       0.57      0.56      0.56      1500
weighted avg       0.88      0.89      0.89      1500



Precisión general: ha subido a 0.8920. Esto es mucho mejor que el 0.0727 anterior, lo que indica que el modelo clasifica de manera más razonable en general.

Para la clase 'fraude' (1):

Precisión: ha mejorado significativamente a 0.20 (antes era 0.07). Esto significa que, de las transacciones que el modelo predice como fraude, el 20% son realmente fraudulentas, reduciendo la cantidad de falsos positivos.
Recall (Sensibilidad): ha disminuido a 0.17 (antes era 1.00). Esto significa que el modelo ahora detecta el 17% de todos los fraudes reales. Estamos perdiendo más fraudes (falsos negativos) en comparación con el umbral anterior.
F1-score: ha subido a 0.18 (antes era 0.14), lo que sugiere un mejor equilibrio entre precisión y recall para la clase minoritaria.

Para la clase 'no fraude' (0):

Las métricas de precisión, recall y F1-score son ahora mucho más altas (alrededor de 0.94-0.95), lo que indica que el modelo está clasificando correctamente la mayoría de las transacciones legítimas.